# Nepali Number Plate YOLOv8 Train and Predict

In [ ]:
!rm -rf *

In [ ]:
%pip install ultralytics

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import shutil
from shutil import copyfile
import matplotlib.pyplot as plt
from PIL import Image
import random
import ultralytics
from ultralytics import YOLO
ultralytics.checks()

In [ ]:
img_path='/kaggle/input/nepali-vehicles-number-plate-dataset/open_source_nepali_plates/images'
label_path='/kaggle/input/nepali-vehicles-number-plate-dataset/open_source_nepali_plates/labels'

In [ ]:
ipaths0=[]
types=[]
for dirname, _, filenames in os.walk(img_path):
    for filename in filenames:
        ipaths0+=[os.path.join(dirname, filename)]
        types+=[filename.split('.')[-1]]
tpaths0=[]
for dirname, _, filenames in os.walk(label_path):
    for filename in filenames:
        tpaths0+=[os.path.join(dirname, filename)]    
ipaths0=sorted(ipaths0)
tpaths0=sorted(tpaths0)
paths=[]
for ip,tp in zip(ipaths0,tpaths0):
    paths+=[(ip,tp)]
random.shuffle(paths)
ipaths=[]
tpaths=[]
for p in paths[0:200]:
    ipaths+=[p[0]]
    tpaths+=[p[1]]
print(len(ipaths))

In [ ]:
print(set(types))
for typei in list(set(types)):
    print(typei,types.count(typei))

# Check Annotation Data

In [ ]:
boxdata=[]
boxfile=[]
for i in range(len(tpaths)):
    tfile=tpaths[i]
    ifile=ipaths[i]
    boxdata+=[np.loadtxt(tfile)]     
    boxfile+=[ifile.split('/')[-1]]
print(boxdata[0])

In [ ]:
BOX=pd.DataFrame()

for i in range(len(boxdata)):
    if type(boxdata[i][0])==np.float64:
        add=pd.DataFrame([boxdata[i]])
        add[5]=boxfile[i]
        BOX=pd.concat([BOX,add])
    else:
        add=pd.DataFrame(boxdata[i])
        add[5]=boxfile[i]
        BOX=pd.concat([BOX,add])       

BOX2=BOX.reset_index(drop=True)
BOX2.iloc[:,0]=BOX2.iloc[:,0].astype(int)
display(BOX2)

In [ ]:
display(BOX2.iloc[:,0].value_counts())

In [ ]:
BOX2[5].value_counts()

In [ ]:
def draw_box(n0):
    
    ipath=ipaths[n0]
    image=cv2.imread(ipath)
    H,W=image.shape[0],image.shape[1]
    file=ipath.split('/')[-1] 
    
    if BOX2[BOX2[5]==file] is not None:
        box=BOX2[BOX2[5]==file]
        box=box.reset_index(drop=True)
        #display(box)
        
        for i in range(len(box)):
            label=int(box.loc[i,0])
            x=box.loc[i,1]
            y=box.loc[i,2]
            w=box.loc[i,3] 
            h=box.loc[i,4]
            x1=((x-w/2)*W).astype(int)
            y1=((y-h/2)*H).astype(int)
            x2=((x+w/2)*W).astype(int)
            y2=((y+h/2)*H).astype(int)
            
            cv2.rectangle(image,(x1,y1),(x2,y2),(0,255,0),1) #green

            #cv2.putText(image, f'{label}', (50,50), 
            #            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)     
    #plt.imshow(image)
    #plt.show()
    
    return image

In [ ]:
images1=[]
for i in range(len(ipaths)):
    images1+=[draw_box(i)]

In [ ]:
from matplotlib import animation, rc
rc('animation', html='jshtml')

In [ ]:
def create_animation(ims):    
    fig=plt.figure(figsize=(10,6))
    #plt.axis('off')
    im=plt.imshow(cv2.cvtColor(ims[0],cv2.COLOR_BGR2RGB))
    plt.close()    
    def animate_func(i):
        im.set_array(cv2.cvtColor(ims[i],cv2.COLOR_BGR2RGB))
        return [im]
    return animation.FuncAnimation(fig, animate_func, frames=len(ims), interval=1000//2)

In [ ]:
create_animation(images1)

# Split Train, Valid and Test

In [ ]:
os.makedirs('datasets', exist_ok=True)
os.makedirs('datasets/train', exist_ok=True)
os.makedirs('datasets/valid', exist_ok=True)
os.makedirs('datasets/test', exist_ok=True)

In [ ]:
for i in range(len(ipaths)):
    ipath=ipaths[i]
    ifile=ipath.split('/')[-1]
    tpath=tpaths[i]
    tfile=tpath.split('/')[-1]

    if i%3==0:
        copyfile(ipath, f'datasets/train/{ifile}')
        copyfile(tpath, f'datasets/train/{tfile}')
    elif i%3==1:
        copyfile(ipath, f'datasets/valid/{ifile}')
        copyfile(tpath, f'datasets/valid/{tfile}')
    else:
        copyfile(ipath, f'datasets/test/{ifile}')
        #copyfile(tpath, f'datasets/test/{tfile}')        

# Create yaml file 

In [ ]:
import yaml

face_yaml = dict(
    train ='train',
    val ='valid',
    test='test',
    nc = 1,
    names = ['character']
)

with open('face.yaml', 'w') as outfile:
    yaml.dump(face_yaml, outfile, default_flow_style=True)
    
%cat face.yaml

# Train

In [ ]:
model = YOLO("yolov8n.pt") 

In [ ]:
!yolo task=detect mode=train model=yolov8n.pt data=face.yaml epochs=20 imgsz=640

# Result of Training

In [ ]:
tpaths2=[]
for dirname, _, filenames in os.walk('/kaggle/working/runs/detect/train'):
    for filename in filenames:
        if filename[-4:]=='.png' or filename[-4:]=='.jpg':
            tpaths2+=[(os.path.join(dirname, filename))]
tpaths2=sorted(tpaths2)
print(tpaths2[0])

In [ ]:
for path in tpaths2:
    image = Image.open(path)
    image=np.array(image)
    plt.figure(figsize=(20,10))
    plt.imshow(image)
    plt.show()

# Predict

In [ ]:
best_path0='runs/detect/train/weights/best.pt'
source0='datasets/test'

In [ ]:
ppaths=[]
for dirname, _, filenames in os.walk(source0):
    for filename in filenames:
        ppaths+=[(os.path.join(dirname, filename))]
ppaths=sorted(ppaths)
print(ppaths[0])
print(len(ppaths))

In [ ]:
model2 = YOLO(best_path0)

In [ ]:
!yolo task=detect mode=predict model={best_path0} conf=0.05 source={source0}

# Result of Prediction

In [ ]:
results = model2.predict(source0,conf=0.05)
print(len(results))

In [ ]:
PBOX=pd.DataFrame(columns=range(6))
for i in range(len(results)):
    arri = pd.DataFrame(results[i].boxes.data.cpu()).astype(float)
    #arri = pd.DataFrame(results[i].boxes.data).astype(float)
    path=ppaths[i]
    file=path.split('/')[-1]
    arri=arri.assign(file=file)
    arri=arri.assign(i=i)
    PBOX=pd.concat([PBOX,arri],axis=0)
PBOX.columns=['x1','y1','x2','y2','confidence','class','file','i']
PBOX['class']=PBOX['class'].astype(int)
display(PBOX)

In [ ]:
PBOX=PBOX.reset_index(drop=True)
#display(df)
#idx = df.groupby('i')['confidence'].idxmax()
#df2 = df.loc[idx]
#PBOX = df2.reset_index(drop=True)

In [ ]:
def draw_box2(n0):
    
    ipath=ppaths[n0]
    image=cv2.imread(ipath)
    H,W=image.shape[0],image.shape[1]
    file=ipath.split('/')[-1]
    
    if PBOX[PBOX['file']==file] is not None:
        box=PBOX[PBOX['file']==file]
        box=box.reset_index(drop=True)
        #display(box)

        for i in range(len(box)):
            label=int(box.loc[i,'class'])
            x1=int(box.loc[i,'x1'])
            y1=int(box.loc[i,'y1'])
            x2=int(box.loc[i,'x2']) 
            y2=int(box.loc[i,'y2'])
            #print(label,x,y,x2,y2)
            cv2.rectangle(image,(x1,y1),(x2,y2),(0,255,0),1) #green

            #cv2.putText(image, f'{label}', (50,50), 
            #            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)       
    #plt.imshow(image)
    #plt.show()   
    
    return image

In [ ]:
images2=[]
for i in tqdm(range(len(ppaths))):
    images2+=[draw_box2(i)]

In [ ]:
create_animation(images2)